### Plots and statistical tests for Batch Processing results (WT and Tumor organoids)

In [ ]:
import pandas as pd
from pathlib import Path

from data_analysis_utils import (
    plot_nuclei_volume_distribution, 
    calculate_technical_replicates_mean_values, 
    plot_boxplots_by_features,
    calculate_biological_replicates_mean_values,
    plot_scatter_nuclei_cyto_ratio,
    calculate_statistical_tests
)

### Configure experiment inputs for WT dataset
Set your input directory and analysis parameters.

In [ ]:
# Inclusive min/max nuclei volume used to filter predicted labels
# Set to None to skip label-size filtering
MIN_MAX_NUCLEI_VOLUME = (250,4000)

# Path to the per-label CSV output from batch processing
PER_LABEL_RESULTS_CSV_PATH = "./results/bp_results/20240607 PGE2MSN WT_organoids/per_label_results_20240607 PGE2MSN WT_organoids_MIN_None_MAX_None_CT2_NP0.csv"

In [ ]:
# Extract sample-specific bp_results directory from per-label CSV path
bp_results_directory = Path(PER_LABEL_RESULTS_CSV_PATH).parent

# Read DataFrame with batch processing results from disk
final_df = pd.read_csv(PER_LABEL_RESULTS_CSV_PATH)

# Plot nuclei volume distribution after 1st-99th percentile normalization (without altering final_df)
plot_nuclei_volume_distribution(final_df, bp_results_directory)

In [ ]:
# Filter out rows where volume_nuclei is less than MIN or greater than MAX (based on visual inspection and volume distribution)
if MIN_MAX_NUCLEI_VOLUME is not None:
    final_df_filtered = final_df[(final_df["volume_nuclei"] >= MIN_MAX_NUCLEI_VOLUME[0]) & (final_df["volume_nuclei"] <= MIN_MAX_NUCLEI_VOLUME[1])]
else:
    final_df_filtered = final_df

# Filter out rows where organoid_id is 0 (orphan cells not mapped to any organoid)
final_df_filtered = final_df_filtered[final_df_filtered["organoid_id"] != 0]

In [ ]:
# Group by organoid identifiers and compute per-organoid averages
grouped_df = calculate_technical_replicates_mean_values(
    final_df_filtered,
    PER_LABEL_RESULTS_CSV_PATH,
    MIN_MAX_NUCLEI_VOLUME,
)

plot_boxplots_by_features(
    grouped_df,
    features=["intensity_mean_nuclei", "intensity_mean_cyto", "nuclei_cyto_ratio"],
    bp_results_directory=bp_results_directory,
)

In [ ]:
# Calculate biological replicates mean values
grouped_bio_rep_df = calculate_biological_replicates_mean_values(
    final_df_filtered,
    PER_LABEL_RESULTS_CSV_PATH,
    MIN_MAX_NUCLEI_VOLUME,
)

plot_scatter_nuclei_cyto_ratio(
    grouped_bio_rep_df,
    PER_LABEL_RESULTS_CSV_PATH,
    treatment_order=["BCM", "MSN", "PGE2"],
)
calculate_statistical_tests(grouped_bio_rep_df, bp_results_directory)